# Spotify Lyrics Extractor v8 — `spotify_data.csv` edition

**Input columns** (from `spotify_data.csv`):  
`artist_name, track_name, track_id, popularity, year, genre, danceability, energy,`  
`key, loudness, mode, speechiness, acousticness, instrumentalness, liveness,`  
`valence, tempo, duration_ms, time_signature`

**Lyrics fallback chain (in order):**  
1. **LRCLib** — free, no auth, no rate-limit (`https://lrclib.net/api/get`)  
2. **LibreLyrics (Spotify plugin)** — Spotify-internal synced lyrics (requires `sp_dc` cookie)  
3. **Genius** — via `lyricsgenius` (requires access token)  
4. **syncedlyrics** — scrapes Lrclib + NetEase (no key needed)

In [1]:
# Install dependencies (uncomment if needed)
# %pip install spotifyscraper pandas tqdm lyricsgenius pyarrow syncedlyrics lrclib librelyrics librelyrics-spotify

In [2]:
import os
import re
import time
import random
import inspect
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import quote

from tqdm import tqdm
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import spotify_scraper as sps
from librelyrics import LibreLyrics
import lyricsgenius
import syncedlyrics

# ── Load spotify_data.csv ────────────────────────────────────────────────────
df = pd.read_parquet('spotify_data.parquet')
print(f"Loaded {len(df)} tracks.")
print(f"Columns: {list(df.columns)}")
display(df.head())

Loaded 1159764 tracks.
Columns: ['Unnamed: 0', 'artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature']


,Unnamed: 0,artist_name,track_name,track_id,popularity,year,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
0,0,Jason Mraz,I Won't Give Up,53QF56cjZA9RTuuMZDrSA6,68,2012,acoustic,0.483,0.303,4,-10.058,1,0.0429,0.6940,0.000000,0.1150,0.139,133.406,240166,3
1,1,Jason Mraz,93 Million Miles,1s8tP3jP4GZcyHDsjvw218,50,2012,acoustic,0.572,0.454,3,-10.286,1,0.0258,0.4770,0.000014,0.0974,0.515,140.182,216387,4
2,2,Joshua Hyslop,Do Not Let Me Go,7BRCa8MPiyuvr2VU3O9W0F,57,2012,acoustic,0.409,0.234,3,-13.711,1,0.0323,0.3380,0.000050,0.0895,0.145,139.832,158960,4
3,3,Boyce Avenue,Fast Car,63wsZUhUZLlh1OsyrZq7sz,58,2012,acoustic,0.392,0.251,10,-9.845,1,0.0363,0.8070,0.000000,0.0797,0.508,204.961,304293,4
4,4,Andrew Belle,Sky's Still Blue,6nXIYClvJAfi6ujLiKqEq8,54,2012,acoustic,0.430,0.791,6,-5.419,0,0.0302,0.0726,0.019300,0.1100,0.217,171.864,244320,4


In [3]:
# ── Credentials (replace with your own values or set as env vars) ───────────
import os

# Spotify sp_dc cookie for LibreLyrics Spotify plugin
os.environ['SPOTIFY_SP_DC'] = 'AQDbkRflBkFLUjL9qtjZmaYeP-tl-MyHIrClz0d_sjRbAcbMUHVEu-2JAOdCOcLu3bPs_SbtXqRNfHNzCEDAhfJaVEIypTnbqGum5SmJ8voKLu1q0qrFy7lVUAWPyiSOAoxapUjs_FkXDZEsQ747cIKmvN_Ei2fgsPtru_qWKRt6NHRMf1LmXtohu24bnyWZ0quY4Q37roeVgGEiJMQ'
# AQBEmTDAxmXyLoRxx-_5050ay2eGEBkp4C92wZxLZuB6tS4R9gfDIsQD0kjw0XvBt9p8JFYK1ummo3JQHnLVkxDEwJeNTMo4lobqm-Kdz_hgHs_owG2dcGPLwsYyB_rkAnlQr0vZ9ptQkN-wT8SPlZo12U-7WqWsc-k5Uj7Im1icnIxxbDm7K5CRckkeiG33NFz2o4uQeBMrSEuYJsQ


# Genius API token
os.environ['GENIUS_ACCESS_TOKEN'] = 'tKtPIQuhJTnYFPEMi4WYwr73Dvcdeiw8taZibAxVFCrWZFE-0jl6g0m0xtWjJKfQ'

In [4]:
# ── Configuration ────────────────────────────────────────────────────────────

# MAX_WORKERS           = 8      # increase to 16-24 if APIs allow
# REQUEST_TIMEOUT       = 10
# MAX_RETRIES           = 4
# BACKOFF_BASE_SECONDS  = 1.4
# JITTER_SECONDS        = 0.35
# LRCLIB_SLEEP_SECONDS  = 0.15   # LRCLib has no rate limit, but be polite
# LIBRELYRICS_SLEEP_SECONDS = 0.28
# GENIUS_SLEEP_SECONDS  = 0.45
# SYNCED_SLEEP_SECONDS  = 0.35

MAX_WORKERS           = 16      # increase to 16-24 if APIs allow
REQUEST_TIMEOUT       = 10
MAX_RETRIES           = 2
BACKOFF_BASE_SECONDS  = 1.0
JITTER_SECONDS        = 0.10
LRCLIB_SLEEP_SECONDS  = 0.05   # LRCLib has no rate limit, but be polite
LIBRELYRICS_SLEEP_SECONDS = 0.10
GENIUS_SLEEP_SECONDS  = 0.15
SYNCED_SLEEP_SECONDS  = 0.15

LRCLIB_BASE_URL = "https://lrclib.net/api"
LRCLIB_USER_AGENT = "spotify-lyrics-extractor/8.0 (https://github.com/martin-he543)"

BATCH_SIZE = 1000
BATCH_DIR  = Path('batch_results')
BATCH_DIR.mkdir(parents=True, exist_ok=True)

# ── Shared HTTP session ───────────────────────────────────────────────────────
http_session   = requests.Session()
retry_strategy = Retry(
    total=MAX_RETRIES,
    backoff_factor=0.6,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
http_adapter = HTTPAdapter(
    max_retries=retry_strategy,
    pool_connections=MAX_WORKERS + 4,
    pool_maxsize=MAX_WORKERS + 4
)
http_session.mount('https://', http_adapter)
http_session.mount('http://',  http_adapter)

# ── Spotify metadata client ──────────────────────────────────────────────────
spotify_client = sps.SpotifyClient(log_level='WARNING')

# ── LibreLyrics client (Spotify plugin) ──────────────────────────────────────
sp_dc = os.getenv('SPOTIFY_SP_DC', '').strip()
librelyrics_client = None
if sp_dc and 'REPLACE_WITH_YOUR_SP_DC_COOKIE' not in sp_dc:
    try:
        librelyrics_client = LibreLyrics(
            config={
                'plugins': {
                    'spotify': {
                        'sp_dc': sp_dc,
                        'synced_lyrics': True
                    }
                },
                'synced_lyrics': True,
                'enhanced_lrc': False
            },
            verbose=False
        )
        print('LibreLyrics client initialised.')
    except Exception as e:
        print(f'LibreLyrics init failed: {e}')
else:
    print('LibreLyrics disabled — set SPOTIFY_SP_DC to enable.')

# ── Genius client ─────────────────────────────────────────────────────────────
GENIUS_ACCESS_TOKEN = os.getenv('GENIUS_ACCESS_TOKEN', '').strip()
genius_client = None
if GENIUS_ACCESS_TOKEN and 'REPLACE_WITH_YOUR_GENIUS_TOKEN' not in GENIUS_ACCESS_TOKEN:
    try:
        genius_client = lyricsgenius.Genius(
            GENIUS_ACCESS_TOKEN,
            timeout=REQUEST_TIMEOUT,
            retries=MAX_RETRIES,
            sleep_time=GENIUS_SLEEP_SECONDS
        )
        genius_client.skip_non_songs         = True
        genius_client.excluded_terms         = ['(Remix)', '(Live)']
        genius_client.verbose                = False
        genius_client.remove_section_headers = True
        print('Genius client initialised.')
    except Exception as e:
        print(f'Genius init failed: {e}')
else:
    print('Genius disabled — set GENIUS_ACCESS_TOKEN to enable.')

# ── Thread-safe caches ────────────────────────────────────────────────────────
_cache_lock                 = threading.Lock()
lyrics_cache_by_track       = {}   # track_id → (lyrics, status)
lyrics_cache_by_name_artist = {}   # (name, artist, source) → (lyrics, status)
metadata_cache_by_track     = {}   # track_id → (metadata, status)

# ── Dataset prep ──────────────────────────────────────────────────────────────
sample_size       = 1_159_764
tracks_to_process = df.head(sample_size).copy()

before_dedup      = len(tracks_to_process)
tracks_to_process = tracks_to_process.drop_duplicates(subset=['track_id']).reset_index(drop=True)
print(f"Removed {before_dedup - len(tracks_to_process)} duplicate track_ids.")
print(f"Processing {len(tracks_to_process)} unique tracks in batches of {BATCH_SIZE}.")


LibreLyrics client initialised.
Genius client initialised.
Removed 0 duplicate track_ids.
Processing 1159764 unique tracks in batches of 1000.


In [5]:
# ── Smoke-test all four sources on the first track in the dataset ─────────────

test_row          = df.iloc[0]
test_track_id     = str(test_row['track_id'])
test_track_name   = str(test_row['track_name'])
test_artist_name  = str(test_row['artist_name'])
test_duration_ms  = int(test_row['duration_ms']) if pd.notna(test_row.get('duration_ms')) else None
test_duration_s   = round(test_duration_ms / 1000) if test_duration_ms else None
test_track_url    = f"https://open.spotify.com/track/{test_track_id}"

print(f"Track : {test_track_name}")
print(f"Artist: {test_artist_name}")
print(f"ID    : {test_track_id}")
print(f"dur   : {test_duration_s}s")
print()

# 1) LRCLib ── primary source ─────────────────────────────────────────────────
lrclib_lyrics = None
lrclib_status = 'skipped'
try:
    params = {
        'track_name':  test_track_name,
        'artist_name': test_artist_name,
    }
    if test_duration_s:
        params['duration'] = test_duration_s
    resp = http_session.get(
        f"{LRCLIB_BASE_URL}/get",
        params=params,
        headers={'User-Agent': LRCLIB_USER_AGENT},
        timeout=REQUEST_TIMEOUT
    )
    if resp.status_code == 200:
        payload = resp.json()
        text = payload.get('plainLyrics') or ''
        if text.strip():
            lrclib_lyrics = text.strip()
            lrclib_status = 'ok_lrclib'
        elif payload.get('instrumental'):
            lrclib_status = 'instrumental_lrclib'
        else:
            lrclib_status = 'not_found_lrclib'
    elif resp.status_code == 404:
        lrclib_status = 'not_found_lrclib'
    else:
        lrclib_status = f'http_{resp.status_code}_lrclib'
except Exception as e:
    lrclib_status = f'error_lrclib: {e.__class__.__name__}'

# 2) LibreLyrics (Spotify plugin) ─────────────────────────────────────────────
librelyrics_lyrics = None
librelyrics_status = 'skipped_no_auth_or_plugin'
if librelyrics_client:
    try:
        response = librelyrics_client.fetch(test_track_url)
        lines = [line.text.strip() for line in response.lyrics if getattr(line, 'text', '').strip()]
        if lines:
            librelyrics_lyrics = "\n".join(lines)
            librelyrics_status = 'ok_librelyrics'
        else:
            librelyrics_status = 'empty_librelyrics'
    except Exception as e:
        librelyrics_status = f'error_librelyrics: {e.__class__.__name__}'

# 3) Genius ── third source ───────────────────────────────────────────────────
genius_lyrics = None
genius_status = 'skipped_no_genius'
if genius_client:
    try:
        song = genius_client.search_song(test_track_name, test_artist_name or None)
        if song and getattr(song, 'lyrics', None) and song.lyrics.strip():
            genius_lyrics = song.lyrics.strip()
            genius_status = 'ok_genius'
        else:
            genius_status = 'not_found_genius'
    except Exception as e:
        genius_status = f'error_genius: {e.__class__.__name__}'

# 4) syncedlyrics ── last resort ──────────────────────────────────────────────
synced_lyrics = None
synced_status = 'skipped'
try:
    search_term = f"{test_track_name} {test_artist_name}".strip()
    sig    = inspect.signature(syncedlyrics.search)
    kwargs = {}
    if 'allow_plain_format' in sig.parameters:
        kwargs['allow_plain_format'] = True
    if 'providers' in sig.parameters:
        kwargs['providers'] = ['Lrclib', 'NetEase']
    raw = syncedlyrics.search(search_term, **kwargs)
    if raw and raw.strip():
        plain = re.sub(r'\[\d{2}:\d{2}\.\d+\]', '', raw).strip()
        synced_lyrics = plain or None
        synced_status = 'ok_synced' if plain else 'empty_synced'
    else:
        synced_status = 'not_found_synced'
except Exception as e:
    synced_status = f'error_synced: {e.__class__.__name__}'

summary = pd.DataFrame([
    {'source': 'lrclib',       'status': lrclib_status, 'chars': len(lrclib_lyrics)      if lrclib_lyrics      else 0},
    {'source': 'librelyrics',  'status': librelyrics_status, 'chars': len(librelyrics_lyrics) if librelyrics_lyrics else 0},
    {'source': 'genius',       'status': genius_status, 'chars': len(genius_lyrics)      if genius_lyrics      else 0},
    {'source': 'syncedlyrics', 'status': synced_status, 'chars': len(synced_lyrics)      if synced_lyrics      else 0},
])
display(summary)

preview = lrclib_lyrics or librelyrics_lyrics or genius_lyrics or synced_lyrics
if preview:
    print("\nPreview (first 600 chars):")
    print(preview[:600])
else:
    print("\nNo lyrics found from any source for this track.")

Track : I Won't Give Up
Artist: Jason Mraz
ID    : 53QF56cjZA9RTuuMZDrSA6
dur   : 240s



,source,status,chars
0,lrclib,ok_lrclib,1430
1,librelyrics,ok_librelyrics,1424
2,genius,error_genius: AssertionError,0
3,syncedlyrics,ok_synced,1465



Preview (first 600 chars):
When I look into your eyes
It's like watching the night sky
Or a beautiful sunrise
Well, there's so much they hold

And just like them old stars
I see that you've come so far
To be right where you are
How old is your soul?

Well, I won't give up on us
Even if the skies get rough
I'm giving you all my love
I'm still looking up

And when you're needing your space
To do some navigating
I'll be here patiently waiting
To see what you find

'Cause even the stars, they burn
Some even fall to the earth
We got a lot to learn
God knows we're worth it
No, I won't give up

I don't wanna be someone who wal


In [6]:
# ── Helper functions ─────────────────────────────────────────────────────────

def polite_pause(base_seconds):
    time.sleep(base_seconds + random.uniform(0, JITTER_SECONDS))


_GENIUS_NO_RETRY = (AssertionError, ValueError, TypeError)

def with_retry(fn, max_attempts=MAX_RETRIES, base_wait=BACKOFF_BASE_SECONDS,
               no_retry_exceptions=()):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(), None
        except no_retry_exceptions as e:
            return None, e
        except Exception as e:
            last_error = e
            if attempt == max_attempts:
                break
            wait_s = (base_wait ** attempt) + random.uniform(0, JITTER_SECONDS)
            time.sleep(wait_s)
    return None, last_error


# ── Source 1: LRCLib (primary — no auth, no rate-limit) ──────────────────────

def fetch_lyrics_from_lrclib(track_name, artist_name, duration_ms=None):
    """
    Query https://lrclib.net/api/get with track_name + artist_name.
    Optionally pass duration_ms (from the dataset) for a more precise match.
    Returns (lyrics_str | None, status_str).
    """
    if not isinstance(track_name, str) or not track_name.strip():
        return None, 'skipped_bad_input_lrclib'

    primary_artist = ''
    if isinstance(artist_name, str) and artist_name.strip():
        primary_artist = artist_name.split(',')[0].strip()

    duration_s = round(int(duration_ms) / 1000) if duration_ms and pd.notna(duration_ms) else None

    cache_key = (track_name.strip().lower(), primary_artist.lower(), 'lrclib')
    with _cache_lock:
        if cache_key in lyrics_cache_by_name_artist:
            return lyrics_cache_by_name_artist[cache_key]

    def _call():
        polite_pause(LRCLIB_SLEEP_SECONDS)
        params = {
            'track_name':  track_name.strip(),
            'artist_name': primary_artist,
        }
        if duration_s:
            params['duration'] = duration_s
        return http_session.get(
            f"{LRCLIB_BASE_URL}/get",
            params=params,
            headers={'User-Agent': LRCLIB_USER_AGENT},
            timeout=REQUEST_TIMEOUT
        )

    resp, err = with_retry(_call)
    if err is not None:
        result = (None, f'error_lrclib: {err.__class__.__name__}')
    elif resp.status_code == 404:
        result = (None, 'not_found_lrclib')
    elif resp.status_code != 200:
        result = (None, f'http_{resp.status_code}_lrclib')
    else:
        try:
            payload = resp.json()
        except Exception:
            result = (None, 'bad_json_lrclib')
        else:
            text = payload.get('plainLyrics') or ''
            if text.strip():
                result = (text.strip(), 'ok_lrclib')
            elif payload.get('instrumental'):
                result = (None, 'instrumental_lrclib')
            else:
                result = (None, 'not_found_lrclib')

    with _cache_lock:
        lyrics_cache_by_name_artist[cache_key] = result
    return result


# ── Source 2: LibreLyrics (Spotify plugin) ───────────────────────────────────

def fetch_lyrics_from_librelyrics(track_id):
    if librelyrics_client is None:
        return None, 'skipped_no_auth_or_plugin'

    cache_key = str(track_id)
    with _cache_lock:
        if cache_key in lyrics_cache_by_track:
            return lyrics_cache_by_track[cache_key]

    track_url = f"https://open.spotify.com/track/{track_id}"

    def _call():
        polite_pause(LIBRELYRICS_SLEEP_SECONDS)
        return librelyrics_client.fetch(track_url)

    response, err = with_retry(_call)
    if err is not None:
        result = (None, f'error_librelyrics: {err.__class__.__name__}')
    else:
        try:
            lines = [
                line.text.strip()
                for line in getattr(response, 'lyrics', [])
                if getattr(line, 'text', '').strip()
            ]
            text = "\n".join(lines).strip()
            result = (text, 'ok_librelyrics') if text else (None, 'empty_librelyrics')
        except Exception as e:
            result = (None, f'parse_error_librelyrics: {e.__class__.__name__}')

    with _cache_lock:
        lyrics_cache_by_track[cache_key] = result
    return result


# ── Source 3: Genius ─────────────────────────────────────────────────────────

def fetch_lyrics_from_genius(track_name, artist_name):
    if genius_client is None:
        return None, 'skipped_no_genius'
    if not isinstance(track_name, str) or not track_name.strip():
        return None, 'skipped_bad_input_genius'

    primary_artist = None
    if isinstance(artist_name, str) and artist_name.strip():
        primary_artist = artist_name.split(',')[0].strip()

    cache_key = (track_name.strip().lower(), (primary_artist or '').lower(), 'genius')
    with _cache_lock:
        if cache_key in lyrics_cache_by_name_artist:
            return lyrics_cache_by_name_artist[cache_key]

    def _call():
        polite_pause(GENIUS_SLEEP_SECONDS)
        return genius_client.search_song(track_name, primary_artist)

    song, err = with_retry(_call, no_retry_exceptions=_GENIUS_NO_RETRY)
    if err is not None:
        err_label = 'bot_block_genius' if isinstance(err, AssertionError) else 'error_genius'
        result = (None, err_label)
    elif song and getattr(song, 'lyrics', None) and song.lyrics.strip():
        result = (song.lyrics.strip(), 'ok_genius')
    else:
        result = (None, 'not_found_genius')

    with _cache_lock:
        lyrics_cache_by_name_artist[cache_key] = result
    return result


# ── Source 4: syncedlyrics (Lrclib + NetEase, no key) ────────────────────────

def fetch_lyrics_from_syncedlyrics(track_name, artist_name):
    if not isinstance(track_name, str) or not track_name.strip():
        return None, 'skipped_bad_input_synced'

    primary_artist = ''
    if isinstance(artist_name, str) and artist_name.strip():
        primary_artist = artist_name.split(',')[0].strip()

    search_term = f"{track_name.strip()} {primary_artist}".strip()
    cache_key   = (search_term.lower(), 'synced')
    with _cache_lock:
        if cache_key in lyrics_cache_by_name_artist:
            return lyrics_cache_by_name_artist[cache_key]

    def _call():
        polite_pause(SYNCED_SLEEP_SECONDS)
        sig    = inspect.signature(syncedlyrics.search)
        kwargs = {}
        if 'allow_plain_format' in sig.parameters:
            kwargs['allow_plain_format'] = True
        if 'providers' in sig.parameters:
            kwargs['providers'] = ['Lrclib', 'NetEase']
        return syncedlyrics.search(search_term, **kwargs)

    raw, err = with_retry(_call)
    if err is not None:
        result = (None, f'error_synced: {err.__class__.__name__}')
    elif raw and raw.strip():
        plain  = re.sub(r'\[\d{2}:\d{2}\.\d+\]', '', raw).strip()
        result = (plain if plain else None, 'ok_synced' if plain else 'empty_synced')
    else:
        result = (None, 'not_found_synced')

    with _cache_lock:
        lyrics_cache_by_name_artist[cache_key] = result
    return result


# ── Metadata helper ───────────────────────────────────────────────────────────

def fetch_track_metadata(track_url, track_id):
    cache_key = str(track_id)
    with _cache_lock:
        if cache_key in metadata_cache_by_track:
            return metadata_cache_by_track[cache_key]

    def _call():
        polite_pause(LIBRELYRICS_SLEEP_SECONDS)
        return spotify_client.get_track_info(track_url)

    metadata, err = with_retry(_call)
    result = (metadata, 'ok') if isinstance(metadata, dict) else (None, 'error')

    with _cache_lock:
        metadata_cache_by_track[cache_key] = result
    return result


# ── Batch flush ───────────────────────────────────────────────────────────────

def flush_batch(rows, batch_index):
    if not rows:
        return None
    batch_df     = pd.DataFrame(rows)
    parquet_path = BATCH_DIR / f'results_batch_{batch_index:05d}.parquet'
    try:
        batch_df.to_parquet(parquet_path, index=False)
        return str(parquet_path)
    except Exception:
        pickle_path = BATCH_DIR / f'results_batch_{batch_index:05d}.pkl'
        batch_df.to_pickle(pickle_path)
        return str(pickle_path)


# ── Per-track worker ──────────────────────────────────────────────────────────

def process_track(row):
    """
    Fetch metadata + lyrics for one track from spotify_data.csv.
    Fallback chain: LRCLib → LibreLyrics → Genius → syncedlyrics
    """
    track_id    = row['track_id']
    track_name  = row.get('track_name', '')
    artist_name = row.get('artist_name', '')
    duration_ms = row.get('duration_ms')
    track_url   = f"https://open.spotify.com/track/{track_id}"

    track_info = {
        'track_id':       track_id,
        'original_name':  track_name,
        'original_artist': artist_name,
    }

    metadata, metadata_status = fetch_track_metadata(track_url, track_id)
    if metadata:
        track_info.update(metadata)
    track_info['metadata_status'] = metadata_status

    lrclib_lyrics, lrclib_status = fetch_lyrics_from_lrclib(track_name, artist_name, duration_ms)
    if lrclib_lyrics:
        track_info['lyrics']        = lrclib_lyrics
        track_info['lyrics_status'] = lrclib_status
        return track_info

    libre_lyrics, libre_status = fetch_lyrics_from_librelyrics(track_id)
    if libre_lyrics:
        track_info['lyrics']        = libre_lyrics
        track_info['lyrics_status'] = f"{lrclib_status}->{libre_status}"
        return track_info

    genius_lyrics, genius_status = fetch_lyrics_from_genius(track_name, artist_name)
    if genius_lyrics:
        track_info['lyrics']        = genius_lyrics
        track_info['lyrics_status'] = f"{lrclib_status}->{libre_status}->{genius_status}"
        return track_info

    synced_lyrics, synced_status = fetch_lyrics_from_syncedlyrics(track_name, artist_name)
    track_info['lyrics'] = synced_lyrics
    track_info['lyrics_status'] = f"{lrclib_status}->{libre_status}->{genius_status}->{synced_status}"
    return track_info


print("Helper functions defined. Ready to run extraction.")

Helper functions defined. Ready to run extraction.


In [7]:
# ── Resume: skip already-processed track IDs ─────────────────────────────────

existing_batch_paths = sorted(BATCH_DIR.glob('results_batch_*'))
processed_track_ids  = set()

for path in existing_batch_paths:
    try:
        if path.suffix == '.parquet':
            batch_df = pd.read_parquet(path, columns=['track_id'])
        elif path.suffix == '.pkl':
            batch_df = pd.read_pickle(path)[['track_id']]
        else:
            continue
        processed_track_ids.update(batch_df['track_id'].dropna().astype(str).tolist())
    except Exception as e:
        print(f"Skipping unreadable batch file {path.name}: {e}")

tracks_to_process['track_id'] = tracks_to_process['track_id'].astype(str)
remaining_tracks = (
    tracks_to_process[~tracks_to_process['track_id'].isin(processed_track_ids)]
    .sort_values(by='popularity', ascending=False)
    .reset_index(drop=True)
)
start_batch_index = len(existing_batch_paths) + 1

print(f"Found {len(existing_batch_paths)} existing batch files "
      f"with {len(processed_track_ids)} processed track_ids.")
print(f"Remaining tracks to process: {len(remaining_tracks)}")

Found 411 existing batch files with 411000 processed track_ids.
Remaining tracks to process: 748764


In [ ]:
# ── Concurrent extraction loop ────────────────────────────────────────────────

if remaining_tracks.empty:
    print("No remaining tracks. Extraction already complete for current sample.")
else:
    rows_iter   = list(remaining_tracks.itertuples(index=False))
    batch_rows  = []
    batch_files = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_idx = {
            executor.submit(process_track, row._asdict()): idx
            for idx, row in enumerate(rows_iter, start=1)
        }

        with tqdm(total=len(future_to_idx), unit='track', dynamic_ncols=True) as pbar:
            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    track_info = future.result()
                except Exception as exc:
                    track_info = {
                        'track_id':       rows_iter[idx - 1]._asdict().get('track_id', 'unknown'),
                        'lyrics_status':  f'exception: {exc.__class__.__name__}',
                    }

                batch_rows.append(track_info)
                pbar.update(1)

                if len(batch_rows) % 10 == 0:
                    tid     = track_info.get('track_id', '?')
                    name    = track_info.get('original_name', '?')
                    artist  = track_info.get('original_artist', '?')
                    lstatus = track_info.get('lyrics_status', '?')
                    mstatus = track_info.get('metadata_status', '?')
                    preview = (track_info.get('lyrics') or '')[:80].replace('\n', ' ')
                    tqdm.write(
                        f"[#{len(batch_rows):>6}] {tid} | {name[:30]:<30} | {artist[:25]:<25} "
                        f"| meta={mstatus:<5} | lyrics={lstatus:<45} | '{preview}'"
                    )

                if len(batch_rows) % BATCH_SIZE == 0:
                    batch_index = start_batch_index + (len(batch_rows) // BATCH_SIZE) - 1
                    batch_file  = flush_batch(batch_rows[-BATCH_SIZE:], batch_index)
                    if batch_file:
                        batch_files.append(batch_file)
                        tqdm.write(f"Saved checkpoint: {batch_file}")

    remainder = batch_rows[-(len(batch_rows) % BATCH_SIZE) or len(batch_rows):]
    if remainder:
        batch_index = start_batch_index + len(batch_files)
        batch_file  = flush_batch(remainder, batch_index)
        if batch_file:
            batch_files.append(batch_file)
            print(f"Saved final checkpoint: {batch_file}")

    print(f"Extraction complete. Checkpoints written this run: {len(batch_files)}")

  0%|          | 20/748764 [00:00<108:19:50,  1.92track/s]

[#    10] 2kybBXh9ehEnjOyrikCXTr | Ella No Lo Supero              | Abraham Vazquez           | meta=ok    | lyrics=ok_lrclib                                     | 'Le marcaba y no contestaba el celular Y abrazó su almohada y comenzó a llorar Y '
[#    20] 6OrvJWyr91g2wY03gpz3nE | Me Dicen Señor                 | Virlan Garcia             | meta=ok    | lyrics=ok_lrclib                                     | 'El poder que traigo no carga cualquiera Que enseñé en las armas y las carrillera'


  0%|          | 30/748764 [00:04<36:17:01,  5.73track/s] 

[#    30] 3YBix3fQId9Lg6glZu3EoP | Voz Da Lapa Feat. Toco         | Rosalia De Souza          | meta=ok    | lyrics=not_found_lrclib->error_librelyrics: LyricsNotFound->not_found_genius->not_found_synced | ''


  0%|          | 40/748764 [00:06<45:03:13,  4.62track/s]

...

  0%|          | 50/748764 [00:09<58:49:06,  3.54track/s]

[#    50] 0c8rwwbWUNocXzFzZyPfDF | Oft gefragt - Live             | AnnenMayKantereit         | meta=ok    | lyrics=ok_lrclib                                     | 'Du hast mich angezogen, ausgezogen, großgezogen Und wir sind umgezogen, ich hab '


  0%|          | 60/748764 [00:12<55:06:28,  3.77track/s]

[#    60] 1C2dNli8xK5O0OHs5R5eM3 | Short Line                     | Pretty Lights             | meta=ok    | lyrics=not_found_lrclib->error_librelyrics: LyricsNotFound->not_found_genius->not_found_synced | ''


  0%|          | 66/748764 [00:14<46:39:26,  4.46track/s]


In [ ]:
# ── Rebuild full results from checkpoint files ────────────────────────────────

batch_paths  = sorted(BATCH_DIR.glob('results_batch_*'))
batch_frames = []

for path in batch_paths:
    if path.suffix == '.parquet':
        batch_frames.append(pd.read_parquet(path))
    elif path.suffix == '.pkl':
        batch_frames.append(pd.read_pickle(path))

results_df = pd.concat(batch_frames, ignore_index=True) if batch_frames else pd.DataFrame()
print(f"Loaded {len(results_df)} extracted rows from {len(batch_frames)} batch files.")

In [ ]:
# ── Merge with original spotify_data.csv and save ────────────────────────────

final_df = pd.merge(tracks_to_process, results_df, on='track_id', how='left')


def coalesce_suffix_columns(df):
    bases = sorted({
        col[:-2] for col in df.columns
        if col.endswith('_x') and f"{col[:-2]}_y" in df.columns
    })
    for base in bases:
        df[base] = df[f"{base}_x"].combine_first(df[f"{base}_y"])
        df = df.drop(columns=[f"{base}_x", f"{base}_y"])
    return df


final_df = coalesce_suffix_columns(final_df)
final_df = final_df.loc[:, ~final_df.columns.duplicated()]

if 'Unnamed: 0' in final_df.columns:
    final_df = final_df.drop(columns=['Unnamed: 0'])

# Preserve all original spotify_data.csv columns + add lyrics columns
ORIGINAL_COLS = [
    'artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre',
    'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
    'duration_ms', 'time_signature',
]
extra_cols = [c for c in final_df.columns if c not in ORIGINAL_COLS]
ordered_cols = [c for c in ORIGINAL_COLS if c in final_df.columns] + extra_cols
final_df = final_df[ordered_cols]

parquet_output = 'spotify_data_with_lyrics.parquet'
csv_output     = 'spotify_data_with_lyrics.csv'

try:
    final_df.to_parquet(parquet_output, index=False)
    print(f"Saved parquet → '{parquet_output}'")
except Exception as e:
    print(f"Parquet save failed ({e}); writing CSV only.")

final_df.to_csv(csv_output, index=False)
print(f"Saved CSV    → '{csv_output}'")

if 'lyrics_status' in final_df.columns:
    print("\nLyrics status breakdown:")
    print(final_df['lyrics_status'].value_counts(dropna=False).to_string())

show_cols = [c for c in ['track_id', 'track_name', 'artist_name', 'lyrics_status', 'lyrics']
             if c in final_df.columns]
print(f"\nTotal columns: {final_df.shape[1]}")
display(final_df[show_cols].head())